# needtoknow: the load benchmark on another vector store

Runs [`benchmarks/load_bench.py`](https://github.com/muzakkirhussain011/needtoknow/blob/main/benchmarks/load_bench.py) on one store, chosen below: one million MS MARCO passages embedded with BAAI/bge-small-en-v1.5, loaded through needtoknow's backend for that store, then recall and latency by permission selectivity and throughput with concurrent users. [`kaggle_stores.sh`](https://github.com/muzakkirhussain011/needtoknow/blob/main/benchmarks/kaggle_stores.sh) starts the store from its release build, since Kaggle has no Docker. The pgvector run, with the cost of a remote permission service, is in [`kaggle_load.ipynb`](https://github.com/muzakkirhussain011/needtoknow/blob/main/benchmarks/kaggle_load.ipynb).

Notebook settings: **GPU** accelerator (for the embeddings) and **Internet on**. To reuse the embeddings of an earlier run, add that notebook's output as input; the GPU is then not needed.

In [ ]:
STORE = "qdrant"  # qdrant, elasticsearch, opensearch or weaviate
REF = "main"  # the needtoknow branch, tag or commit to benchmark

!rm -rf needtoknow && git clone -q https://github.com/muzakkirhussain011/needtoknow.git
!git -C needtoknow checkout -q {REF} && git -C needtoknow log -1 --format="%H %cs %s"
!pip install -q "./needtoknow[qdrant,weaviate,elasticsearch,opensearch]"

In [ ]:
!bash needtoknow/benchmarks/kaggle_stores.sh {STORE}

In [ ]:
import shlex
import shutil
from pathlib import Path

# Embeddings saved by an earlier run (its output attached as input) skip the GPU step.
for saved in Path("/kaggle/input").glob("**/cache/*.npz"):
    Path("/kaggle/working/cache").mkdir(parents=True, exist_ok=True)
    shutil.copy(saved, "/kaggle/working/cache")
    print(f"reusing {saved}")

urls = {
    "qdrant": "http://127.0.0.1:6333",
    "elasticsearch": "http://127.0.0.1:9200",
    "opensearch": "http://127.0.0.1:9200",
    "weaviate": "http://127.0.0.1:8080",
}
args = [
    "--backend",
    STORE,
    "--url",
    urls[STORE],
    "--dataset",
    "msmarco",
    "--embedder",
    "BAAI/bge-small-en-v1.5",
    "--query-prefix",
    "Represent this sentence for searching relevant passages: ",
    "--cache",
    "/kaggle/working/cache",
    "--out",
    f"/kaggle/working/load_{STORE}.json",
    "--markdown",
    f"/kaggle/working/load_{STORE}.md",
]
command = shlex.join(["python", "needtoknow/benchmarks/load_bench.py", *args])
!{command}

In [ ]:
from pathlib import Path

from IPython.display import Markdown

Markdown(Path(f"/kaggle/working/load_{STORE}.md").read_text())